# Trabajo en clase — Q-Learning con Taxi (Actividades)

Este notebook contiene exclusivamente la actividad y preguntas teóricas para Taxi-v4, junto con sus dependencias base.

## 1. Dependencias y base (`comun.py`)

In [ ]:
import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt

from IPython.display import HTML
from matplotlib import animation

env = gym.make("Taxi-v4", render_mode="rgb_array")

print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)

state, info = env.reset(seed=42)
action = env.action_space.sample()
next_state, reward, terminated, truncated, info = env.step(action)

print("Estado:", state)
print("Acción:", action)
print("Nuevo estado:", next_state)
print("Recompensa:", reward)
print("Finalizado:", terminated)

n_states = env.observation_space.n
n_actions = env.action_space.n

Q = np.zeros((n_states, n_actions))

print("Forma de Q:", Q.shape)
print(Q[:5])



Número de estados: 500
Número de acciones: 6
Estado: 386
Acción: 2
Nuevo estado: 386
Recompensa: -1
Terminated: False
Shape de Q: (500, 6)
[[0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]]


## 2. Preguntas Teóricas

### Pregunta 1

- **¿Cuántos estados y cuántas acciones tiene Taxi-v4?**
> Tiene 500 estados y 6 acciones.
- **Explique brevemente por qué Taxi tiene muchos más estados que FrozenLake.**
> En FrozenLake (4x4) el estado solo depende de la posición en la cuadrícula (16 estados). En Taxi, el estado combina tres elementos: la posición del taxi (25 posibles), la ubicación actual del pasajero (5 posibles) y el destino del pasajero (4 posibles). En total: $25 \times 5 \times 4 = 500$ combinaciones (estados) posibles.

### Pregunta 2

**¿En la interacción anterior identifique $s_t, a_t, r_{t+1}, s_{t+1}$ y qué representa cada elemento?**

- **$s_t$ (`state`)**: El estado actual del entorno antes de realizar la acción.
- **$a_t$ (`action`)**: La acción elegida para ejecutar (ej. moverse en alguna dirección, recoger o dejar).
- **$r_{t+1}$ (`reward`)**: La recompensa obtenida al realizar esa acción desde el estado previo.
- **$s_{t+1}$ (`next_state`)**: El estado resultante en el entorno después de realizar la acción.

###  Pregunta 3
**¿Cuántos valores debe aprender el agente en total?**
> Debe aprender un valor Q por cada par estado-acción. 
  Por lo tanto: $500 \text{ estados} \times 6 \text{ acciones} = 3,000$ valores totales.


## 3. Actividad 1 — Política Epsilon-Greedy

In [ ]:
def choose_action(Q, state, epsilon, env):
    """
    Política epsilon-greedy para seleccionar acciones
    """
    # 1. decidir si explorar o explotar
    if random.random() < epsilon:
        # Explorar: retornar una acción aleatoria
        return env.action_space.sample()
    else:
        # Explotar: retornar la mejor acción para el estado actual según Q
        q_values = Q[state]
        max_q = np.max(q_values)
        
        # En caso de empate entre varias acciones con el mismo valor máximo Q,
        # seleccionamos aleatoriamente entre ellas.
        best_actions = np.flatnonzero(q_values == max_q)
        return int(np.random.choice(best_actions))

# Prueba rápida de la función de Actividad 1
state, info = env.reset(seed=42)
epsilon = 0.1
action = choose_action(Q, state, epsilon, env)
print(f"Probando choose_action en el estado {state} con epsilon {epsilon} -> Acción seleccionada: {action}")


## 4. Actividad 2 — Actualizar Q 

In [ ]:

def update_q(Q, state, action, terminated, reward, next_state, alpha, gamma):
    
    current_q= Q[state, action]
     #si ya terminamos el episodio ya no hay valores de q futuros
    if terminated:
        best_next_q = 0.0
    else:
        best_next_q = np.max(Q[next_state])
        # Q es una matriz de numpy por lo tanto no se utiliza el .values() 
        #basta con solo poner el indice para acceder a la fila de acciones

    #TD TARGET
    td_target = reward + gamma * best_next_q
    #TD ERROR
    td_error = td_target - current_q

    Q[state, action]=( #manera de acceder a el valor que estamos actualizando
        current_q + alpha * td_error
    )


print("Test for updating Q")

print("Transition:")
print(f"{state} --{action} / r={reward}--> {next_state}")

print("\nQ before update:")
print(Q[state])
gamma = 0.9
alpha = 0.1

update_q(
    Q,
    state,
    action,
    reward,
    next_state,
    terminated,
    gamma, 
    alpha

)

print("\nQ after update:")
print(Q[state])


    

## 7. Entrenamiento

Ahora implemente el ciclo completo de Q-Learning.

En cada episodio:

1. reiniciar el ambiente;
2. escoger una acción;
3. ejecutar `env.step(action)`;
4. actualizar $Q(s,a)$;
5. mover el agente a `next_state`;
6. terminar cuando el episodio finalice.

Use inicialmente:

```python
alpha = 0.1
gamma = 0.95
epsilon = 0.1
episodes = 5000
```

### Actividad 3
Complete la función.


In [ ]:
def train_q_learning(
    env,
    Q,
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1,
    max_steps=200
):
    rewards = []

    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):

            # 1: Escoger acción usando la política epsilon-greedy
            action = choose_action(Q, state, epsilon, env)

            # 2: Ejecutar la acción en el ambiente
            next_state, reward, terminated, truncated, _ = env.step(action)

            # 3: Actualizar la Q-table basado en la actividad anerior
            update_q(Q, state, action, terminated, reward, next_state, alpha, gamma)

            # 4: Actualizar el estado actual y sumar la recompensa
            state = next_state
            total_reward += reward

            # 5: Terminar el ciclo si el episodio acabó 
            if terminated or truncated:
                break

        rewards.append(total_reward)

    return Q, rewards

## 8. Entrenar el agente

Ejecute el entrenamiento una vez haya completado las funciones anteriores.


In [ ]:
Q_initial = np.zeros((n_states, n_actions))

Q_trained, rewards = train_q_learning(
    env,
    Q_initial.copy(),
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1
)

# 2. Imprimir lo que guardaron las variables
print("¡Entrenamiento terminado!")
print(f"Se guardaron recompensas de {len(rewards)} episodios.")

print("\nAsí quedó la Q-table (mostrando las primeras 5 filas):")
print(Q_trained[:5])


## 9. Curva de aprendizaje

Observe cómo cambia la recompensa durante el entrenamiento.


In [ ]:
window = 100

moving_average = np.convolve(
    rewards,
    np.ones(window) / window,
    mode="valid"
)

plt.figure(figsize=(10, 4))
plt.plot(moving_average)
plt.xlabel("Episodio")
plt.ylabel("Recompensa promedio")
plt.title(f"Taxi-v3 — recompensa promedio ({window} episodios)")
plt.show()


### Pregunta 4

Describa la curva de aprendizaje.

- ¿La recompensa promedio mejora?
> Sí, la recompensa promedio mejora de forma constante a lo largo de los episodios, subiendo de valores muy negativos (gran cantidad de pasos o penalizaciones) a valores positivos.
- ¿Después de aproximadamente cuántos episodios comienza a estabilizarse?
> La curva empieza a estabilizarse alrededor de los 1500 a 2000 episodios, donde las fluctuaciones se reducen y la recompensa se mantiene cerca del máximo.
- ¿El comportamiento observado indica convergencia perfecta o solamente una política razonablemente buena?
> Indica que se ha alcanzado una política óptima o muy cercana a la óptima, ya que el promedio de recompensas se estabiliza de manera plana en la parte superior del gráfico, maximizando la recompensa final en casi cada episodio.

## 10. Reproducir un episodio

La siguiente función ejecuta una política greedy usando la Q-table aprendida y guarda los frames del episodio.


In [ ]:
def play_episode(env, Q, max_steps=200, seed=None):
    state, _ = env.reset(seed=seed)

    frames = [env.render()]
    total_reward = 0

    for _ in range(max_steps):

        q_values = Q[state]
        max_q = np.max(q_values)

        best_actions = np.flatnonzero(q_values == max_q)
        action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())

        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=500):
    fig = plt.figure(figsize=(6, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


## 11. Comparar antes y después

Primero observe un Taxi sin entrenamiento usando una Q-table en cero.


In [ ]:
frames_initial, reward_initial = play_episode(
    env,
    Q_initial,
    max_steps=50,
    seed=7
)

print("Recompensa total sin entrenamiento:", reward_initial)
frames_to_video(frames_initial, interval=500)


Ahora observe el agente entrenado.


In [ ]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    max_steps=200,
    seed=7
)

print("Recompensa total después del entrenamiento:", reward_trained)
frames_to_video(frames_trained, interval=500)


### Pregunta 5

Compare los dos episodios.

- ¿Qué diferencias observa en el comportamiento del taxi?
> El comportamiento cambia bastante, porque al inicio, como no había explorado, se la pasaba dando vueltas al azar y se devolvía, alejándose de la meta o del lugar donde debía recoger al pasajero. En cambio, ya entrenado, va directo al grano.

- ¿El taxi sin entrenamiento logra completar la tarea?
> No, sin el entrenamiento no lograba completarla, porque en esa exploración perdía mucho tiempo devolviéndose. Como solo teníamos 50 pasos por episodio, se le acababan los intentos antes de poder llegar a la meta.

- ¿El agente entrenado evita acciones innecesarias?
> Así es, evita acciones innecesarias porque ya aprendió que dar pasos en falso le quita puntos. Por eso ya no pierde tiempo en movimientos que no lo acercan a su destino.

- ¿Qué evidencia visual le permite afirmar que el agente aprendió?
> La evidencia visual más clara es que podemos ver al taxi ir directamente a recoger a la persona y llevarla hasta el edificio (la meta) sin quedarse atascado dando vueltas por el mapa.

## 12. Analizar la política aprendida

Seleccione un estado cualquiera y observe los valores aprendidos para sus seis acciones.


In [ ]:
state = 123

print("Estado:", state)
print("Q-values:", Q_trained[state])
print("Mejor acción:", np.argmax(Q_trained[state]))


### Pregunta 6

Para el estado seleccionado:

1. ¿Cuál es la acción con mayor valor Q?
> La acción con mayor valor Q es la número 3 con un valor de 2.71224351. 

2. ¿Qué significa que una acción tenga un valor Q mayor que otra?
> Significa que es la mejor decisión que el agente puede tomar. Al observar la matriz, el agente elige esa acción con el valor mayor porque ya aprendió que las otras opciones no le darán un resultado tan bueno.

3. ¿Por qué no podemos interpretar $Q(s,a)$ únicamente como la recompensa inmediata de ejecutar la acción?
> Porque el valor Q no solo mira los puntos que el agente gana en ese preciso momento, sino que también tiene en cuenta todas las recompensas futuras. Es decir, el agente piensa a largo plazo, a veces dar un paso no da puntos inmediatamente, pero lo pone en la posición correcta para completar la meta más adelante.

## 13. Experimentación

Modifique **solo uno** de los siguientes hiperparámetros y vuelva a entrenar:

- $\alpha$
- $\gamma$
- $\epsilon$

### Pregunta 7

Compare el nuevo entrenamiento con el original.

Explique cómo el cambio del hiperparámetro afectó:

- velocidad de aprendizaje,
- estabilidad,
- recompensa final,
- comportamiento observado.


In [ ]:
Q_initial_exp = np.zeros((n_states, n_actions))

Q_trained_exp, rewards_exp = train_q_learning(
    env,
    Q_initial_exp.copy(),
    episodes=5000,
    alpha=0.8, # Modificado de 0.1 a 0.8
    gamma=0.95,
    epsilon=0.1
)

moving_average_exp = np.convolve(
    rewards_exp,
    np.ones(window) / window,
    mode="valid"
)

plt.figure(figsize=(10, 4))
plt.plot(moving_average, label="Original (alpha=0.1)")
plt.plot(moving_average_exp, label="Experimento (alpha=0.8)")
plt.xlabel("Episodio")
plt.ylabel("Recompensa promedio")
plt.title(f"Comparación de hiperparámetros ({window} episodios)")
plt.legend()
plt.show()


### Respuesta a Pregunta 7

- **velocidad de aprendizaje**: Al aumentar el parámetro $\alpha$ a 0.8, el aprendizaje inicial es más acelerado, subiendo la recompensa promedio más rápido al inicio.
- **estabilidad**: A pesar de aprender más rápido, el modelo es más inestable y la curva puede presentar más fluctuaciones al estar sobreescribiendo fuertemente el conocimiento anterior con cada nueva experiencia.
- **recompensa final**: La recompensa final es muy similar, ya que la política alcanza un nivel óptimo o casi óptimo de cualquier forma.
- **comportamiento observado**: El agente es capaz de aprender a resolver la tarea en menos intentos, aunque los saltos en la estimación de los valores Q hacen que la gráfica no sea tan suave como en el caso original.

## Entrega

El notebook debe contener:

1. implementación de `choose_action`;
2. implementación de `update_q`;
3. implementación de `train_q_learning`;
4. curva de aprendizaje;
5. visualización del agente antes y después del entrenamiento;
6. respuestas a las siete preguntas.

No es necesario modificar las funciones auxiliares de visualización.
